In [39]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt



In [40]:
group_1 = pd.read_csv("C:/Users/Shu/Downloads/group1.csv")
group_2 = pd.read_csv("C:/Users/Shu/Downloads/group2_female.csv") # everything the same but change to female
group_3 = pd.read_csv("C:/Users/Shu/Downloads/group3.csv")
group_4 = pd.read_csv("C:/Users/Shu/Downloads/group4.csv")
group_5 = pd.read_csv("C:/Users/Shu/Downloads/group5.csv")

In [82]:
columns_to_keep = [
    'HYPEV', 'CHLEV', 'CHDEV', 'ANGEV', 'MIEV', 'HRTEV', 'STREV', 'EPHEV',
    'COPDEV', 'AASMEV', 'ULCEV', 'ULCCOLEV', 'CANEV', 'SINYR', 'CBRCHYR',
    'KIDWKYR', 'LIVYR', 'ARTH1', 'VIM_GLEV',
    'AGE_P', 'SEX', 'RACERPI2', 'R_MARITL', 'ONEJOB', 'SMKEV', 'REGION',
    'PAR_STAT', 'WRKCATA', 'AWEIGHTP', 'DIBEV1'
]

# Filter the dataset to keep only the specified columns
group_1 = group_1[columns_to_keep]
group_2 = group_2[columns_to_keep]
group_3 = group_3[columns_to_keep]
group_4 = group_4[columns_to_keep]
group_5 = group_5[columns_to_keep]


In [83]:
group1 = group_1.replace([7,8,9], 0)
group1 = group1.apply(pd.to_numeric, errors='coerce')
group1 = group1.fillna(0)
group1 = group1.replace(2, 0)

group2 = group_2.replace([7,8,9], 0)
group2 = group2.apply(pd.to_numeric, errors='coerce')
group2 = group2.fillna(0)
group2 = group2.replace(2, 0)

group3 = group_3.replace([7,8,9], 0)
group3 = group3.apply(pd.to_numeric, errors='coerce')
group3 = group3.fillna(0)
group3 = group3.replace(2, 0)

group4 = group_4.replace([7,8,9], 0)
group4 = group4.apply(pd.to_numeric, errors='coerce')
group4 = group4.fillna(0)
group4 = group4.replace(2, 0)

group5 = group_5.replace([7,8,9], 0)
group5 = group5.apply(pd.to_numeric, errors='coerce')
group5 = group5.fillna(0)
group5 = group5.replace(2, 0)

In [84]:
# group1.to_csv('group1.csv', index=False)
# group2.to_csv('group2.csv', index=False)

We can try two ways, the first one is to choose several complications, each at a time and predict, we can also create a complication flag to see people who have at least one complication

In [85]:
complication_columns = [
    'HYPEV', 'CHLEV', 'CHDEV', 'ANGEV', 'MIEV', 'HRTEV', 'STREV', 'EPHEV',
    'COPDEV', 'AASMEV', 'ULCEV', 'ULCCOLEV', 'CANEV'
]
group1['Complications_Flag'] = group1[complication_columns].max(axis=1)

# Define features and target for the model
missing_columns = [col for col in complication_columns if col not in group1.columns]
target = 'Complications_Flag'

In [86]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


features = [
    'AGE_P', 'SEX', 'RACERPI2', 'R_MARITL', 'ONEJOB', 'SMKEV', 'REGION',
    'PAR_STAT', 'WRKCATA', 'AWEIGHTP', 'DIBEV1'
]
target = 'Complications_Flag'

X = group1[features]
y = group1[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the logistic regression model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Predict on the test set
y_pred = log_reg.predict(X_test)

# Display the classification report
report = classification_report(y_test, y_pred, output_dict=False)
print(report)

              precision    recall  f1-score   support

           0       0.69      0.92      0.79      1354
           1       0.65      0.26      0.37       754

    accuracy                           0.68      2108
   macro avg       0.67      0.59      0.58      2108
weighted avg       0.68      0.68      0.64      2108



In [87]:
import statsmodels.api as sm

X_train_sm = sm.add_constant(X_train)

# Fit the logistic regression model using statsmodels
logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit()

# Display the summary of the logistic regression model, including p-values and confidence intervals
summary = result.summary()
summary

Optimization terminated successfully.
         Current function value: 0.611244
         Iterations 5


LinAlgError: Singular matrix

In [89]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


X_train_vif = X_train_sm.copy()  # Ensure we're using the training data with a constant added for intercept
vif_data = pd.DataFrame()
vif_data["Feature"] = X_train_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_vif.values, i) for i in range(X_train_vif.shape[1])]

# Display features with high VIF values (typically VIF > 5 indicates multicollinearity)
vif_data.sort_values(by="VIF", ascending=False)

C:\Users\Shu\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


,Feature,VIF
0,const,57.293445
2,SEX,1.247266
10,AWEIGHTP,1.246741
1,AGE_P,1.230182
4,R_MARITL,1.169081
8,PAR_STAT,1.089275
3,RACERPI2,1.033959
11,DIBEV1,1.017944
9,WRKCATA,1.015837
7,REGION,1.012165


In [90]:
# Remove the SMKEV column due to constant values causing multicollinearity issues
X_train_vif_dropped = X_train_vif.drop(columns=['SMKEV'])

# Fit the logistic regression model using statsmodels without the SMKEV column
logit_model = sm.Logit(y_train, X_train_vif_dropped)
result = logit_model.fit()

# Display the summary of the logistic regression model, including p-values and confidence intervals
summary = result.summary()
summary

Optimization terminated successfully.
         Current function value: 0.611244
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:     Complications_Flag   No. Observations:                 4917
Model:                          Logit   Df Residuals:                     4906
Method:                           MLE   Df Model:                           10
Date:                Sun, 03 Nov 2024   Pseudo R-squ.:                 0.06076
Time:                        22:33:41   Log-Likelihood:                -3005.5
converged:                       True   LL-Null:                       -3199.9
Covariance Type:            nonrobust   LLR p-value:                 2.212e-77
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.1351      0.244    -16.917      0.000      -4.614      -3.656
AGE_P          0.0443      0.004     11.470      0.000       0.037       0.052
SEX           -0.2249      0.069     -3.236      0.001      -0.361      -0.089
RACERPI2       0.0016      0.028      0.056      0.955      -0.054       0.057
R_MARITL       0.0496      0.021      2.326      0.020       0.008       0.091
ONEJOB         0.0195      0.114      0.171      0.864      -0.204       0.243
REGION        -0.0042      0.020     -0.211      0.833      -0.044       0.035
PAR_STAT       0.0574      0.031      1.847      0.065      -0.004       0.118
WRKCATA        0.0518      0.033      1.569      0.117      -0.013       0.117
AWEIGHTP       0.0105      0.001     10.196      0.000       0.008       0.013
DIBEV1         0.4886      0.085      5.733      0.000       0.322       0.656
==============================================================================
"""

In [91]:
# only based on hypertension
X = group1[features]
y = group1["HYPEV"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the logistic regression model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Predict on the test set
y_pred = log_reg.predict(X_test)

# Display the classification report
report = classification_report(y_test, y_pred, output_dict=False)
print(report)

              precision    recall  f1-score   support

           0       0.87      1.00      0.93      1835
           1       0.58      0.04      0.08       273

    accuracy                           0.87      2108
   macro avg       0.73      0.52      0.50      2108
weighted avg       0.84      0.87      0.82      2108



In [94]:
X_train_vif_dropped = X_train_vif.drop(columns=['SMKEV'])

# Fit the logistic regression model using statsmodels without the SMKEV column
logit_model = sm.Logit(y_train, X_train_vif_dropped)
result = logit_model.fit()

# Display the summary of the logistic regression model, including p-values and confidence intervals
summary = result.summary()
summary

Optimization terminated successfully.
         Current function value: 0.336056
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  HYPEV   No. Observations:                 4917
Model:                          Logit   Df Residuals:                     4906
Method:                           MLE   Df Model:                           10
Date:                Sun, 03 Nov 2024   Pseudo R-squ.:                  0.1185
Time:                        22:37:27   Log-Likelihood:                -1652.4
converged:                       True   LL-Null:                       -1874.6
Covariance Type:            nonrobust   LLR p-value:                 3.253e-89
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -7.5827      0.375    -20.208      0.000      -8.318      -6.847
AGE_P          0.0688      0.006     11.958      0.000       0.057       0.080
SEX           -0.2505      0.099     -2.540      0.011      -0.444      -0.057
RACERPI2      -0.0654      0.046     -1.436      0.151      -0.155       0.024
R_MARITL       0.0592      0.027      2.152      0.031       0.005       0.113
ONEJOB         0.0463      0.165      0.281      0.779      -0.276       0.369
REGION         0.0350      0.030      1.177      0.239      -0.023       0.093
PAR_STAT      -0.0017      0.044     -0.039      0.969      -0.089       0.085
WRKCATA       -0.0022      0.047     -0.046      0.963      -0.095       0.090
AWEIGHTP       0.0174      0.001     12.083      0.000       0.015       0.020
DIBEV1         0.4647      0.083      5.572      0.000       0.301       0.628
==============================================================================
"""

Here we can see multicolinearity so we did VIF for the data